# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Display metadata summary (access attributes, not as dict or list)
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.date_published}")
print(f"Version: {metadata.version}")
print(f"Cite as: {metadata.cite_as}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets and fields. All `@id` references are maintained for traceability and schema reproducibility.

**Note:** The following cell programmatically lists the available RecordSets and their Fields with their respective `@id`s.

In [ ]:
# List all RecordSets and their fields by @id
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
record_set_ids = []
for rs in record_sets:
    print(f"RecordSet name: {rs.name} | @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (type: {field.data_type_repr}, @id: {field.id})")
    print('')

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for analysis. All record sets are referenced by their `@id`. 

**Note:** This cell loads all records for each RecordSet into a DataFrame, using the RecordSet `@id` as the key. Modify as needed for your analysis focus.

In [ ]:
# Extract data for each record set
# Use the record_set_ids from the previous cell
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show list of columns/fields for the first RecordSet as an example
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_record_set_id is not None:
    print(f"DataFrame columns for RecordSet '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, or grouping. All field accesses are by exact column name (`@id`).

**For demonstration**: Choose the first numeric field from the first record set. **If needed, adapt the field IDs according to your schema.**

In [ ]:
import numpy as np

# Identify main DataFrame and choose a numeric field for demo
record_set_id = main_record_set_id
df = dataframes[record_set_id]
# Find numeric field(s)
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Using numeric field by @id: {numeric_field}")

    # Filtering: keep rows where value > threshold (example: 5th percentile)
    threshold = np.nanpercentile(df[numeric_field], 5)
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (5th percentile):")
    display(filtered_df[[numeric_field]].head())

    # Normalize
    colnorm = f"{numeric_field}_normalized"
    filtered_df[colnorm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, colnorm]].head())

    # Group by another categorical field if available
    group_field = None
    candidate_groups = [col for col in df.columns if col != numeric_field and df[col].dtype == 'object']
    if candidate_groups:
        group_field = candidate_groups[0]
    if group_field is not None:
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
else:
    print("No numeric field found in the first record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

> **Tip:** Use field `@id` in column selection to ensure clarity and schema correctness.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_fields:
    # Histogram of the first numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # If a group field is available, show boxplot
    if group_field is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.xticks(rotation=45)
        plt.title(f'{numeric_field} grouped by {group_field}')
        plt.show()

## 6. Conclusion
This notebook provided an exploration of the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`. It demonstrates how to:
* Load and introspect dataset metadata
* Examine schema record sets and fields by `@id`
* Extract and process records into pandas DataFrames
* Perform basic EDA with normalization and grouping
* Visualize field distributions

Continue your analysis by selecting further fields by `@id` and customizing analytical and visualization workflows as required.